# V20: external validation on CheXpert Plus (Stanford), with clinical history, lateral views and race

Run this **after** V19 has finished, in Colab with the same Google Drive account. It needs the files under `methodology_v19/<hash>/seed0/` (`fold*.npz` and `fold*_bundle_PRIVATE.pkl`). Nothing is retrained. The MIMIC-trained heads, gate models, conformal quantiles and calibration sets are applied **unchanged** to a second hospital.

| Gap left after V19 (Limitations) | What V20 adds |
|---|---|
| The only external cohort (NIH) is image-only and shares 8 labels, which NIH defines differently. | CheXpert Plus provides the report's **clinical history** and the **same 14 CheXpert labels** (uncertain masked, blank negative, the same rule as MIMIC). The risk targets therefore keep the same meaning, and the absolute MIMIC targets can be reused. |
| The lateral-view action could not be tested outside MIMIC. | Uses **lateral views** from the same studies, so ASK-LATERAL and ASK-CONTEXT can be tested at a new hospital. |
| Subgroups on NIH covered sex and age only. | Audits **sex, age band and race/ethnicity** at the external site. |
| Label source. | CheXpert Plus ships CheXbert-style labels as JSON (impression, findings, whole report). The primary labels come from the impression, or from the findings when a report has no impression, which is the rule the MIMIC labeler follows. The whole-report labels serve as a **label-source sensitivity** check. This is not an independent labeler. |

**Order:** 0–1 setup → 2 connect and inspect the schema (catalogue metadata only) → 3 map the schema (check the printed mapping) → 4 build the cohort → 5 encode (resumable) → 6 evaluate → 7 export.

**Time:** encoding about 6,000 frontal images plus their laterals takes about 2–4 hours on the free CPU runtime, or about 30–60 minutes on a T4 GPU. A GPU runtime is fine: the encoder is frozen and float32, so the embeddings differ from the CPU ones only at floating-point level. Everything is checkpointed, so a disconnect resumes where it stopped.

**Data terms:** CheXpert Plus is distributed under the Stanford AIMI research use agreement. Before setting `I_CONFIRM_CHEXPERT_PLUS_TERMS = True`, check that the agreement allows processing in Colab and keeping derived embeddings on your Drive. Files named `*_PRIVATE*` hold patient-level derived data and are never added to the shareable bundle.

> **Honest scope.** These are the same frozen models applied to a different hospital (the exchangeability assumption behind the guarantee is not assured, so the target is checked empirically). The labels are report-derived. Request costs and expert accuracy are assumptions, as in V19. No clinician outcomes.

In [ ]:
from pathlib import Path
import sys, json, hashlib, pickle, time, os, platform, re, warnings, gc
import importlib.util, subprocess
DRIVE_MOUNT = '/content/drive_fresh'
RUN_NAME = 'full_20261015_171819_876684'
RUN_DIR = Path(DRIVE_MOUNT)/'MyDrive/adaptive_risk_gate_clean/runs'/RUN_NAME
V13_REL = 'methodology_v13_final/d8228aba22e5bffd9bc8'     # fresh 2,000-patient cohort (features already cached)
packages = {'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','scipy':'scipy',
            'sklearn':'scikit-learn','openpyxl':'openpyxl','threadpoolctl':'threadpoolctl'}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore')
print('Same source run:', RUN_DIR)
print('V20: reuses the V19 checkpoints; only sections 2-5 contact Redivis / download encoder weights.')

In [ ]:
# Reuse an active mount; mount once if needed; never force-remount.
if not RUN_DIR.is_dir():
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError('Open this notebook in Colab with the same Drive account as the earlier runs.') from exc
    mount = Path(DRIVE_MOUNT)
    if not os.path.ismount(mount):
        if mount.exists() and any(mount.iterdir()):
            raise RuntimeError(f'{mount} is nonempty but not an active Drive mount. Resolve it; nothing was changed.')
        drive.mount(DRIVE_MOUNT)
if not RUN_DIR.is_dir():
    raise FileNotFoundError(f'Completed run not found: {RUN_DIR}')
required = ['persistent_encoded_data_PRIVATE.pkl', 'frozen_manifest.json', 'config.json',
            'encoder_manifest.json', 'cohort_summary.csv']
absent = [n for n in required if not (RUN_DIR/n).is_file()]
if absent:
    raise FileNotFoundError('Saved inputs missing: ' + ', '.join(absent) + '. Nothing is re-extracted.')
print('All required cached inputs present.')

## 0. Fixed helpers (identical to V19)

In [ ]:
from scipy.special import expit, logit, xlogy
from scipy.stats import binom
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score

LABELS = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
          'Pleural Other', 'Fracture', 'Support Devices']
SEVERITY = np.array([0, .45, .45, .5, .65, .8, .7, .8, .4, 1, .6, .35, .5, .3])
SEVERE = SEVERITY >= .7
PARTS = ['train', 'tune', 'gate_fit', 'gate_select', 'calibration', 'test']
EPS = 1e-6
CUTOFF = 0.5

# ------------------------------------------------------------------ fixed analysis plan (written BEFORE any result)
PLAN = dict(
    method='ARG_V19_NEGBIO_SUBGROUPS_NIH',
    folds=6,                       # patient-grouped outer folds: fit(K-3) / dev / cal / eval
    seed=20261019,
    fp_weight=1.0,                 # same development assumption as V4-V6
    conformal_alpha=0.10,
    delta=0.10,                    # LTT failure probability
    harm_kinds={'fn': [0.01, 0.02, 0.03, 0.04],            # severity-weighted missed-finding burden per case
                'severe': [0.05, 0.10, 0.15, 0.20]},        # critical-finding missed-escalation rate
    budgets=[0.1, 0.2, 0.3, 0.4],  # same expected review budgets as earlier notebooks
    primary_budget=0.3,
    boot=5000,
    seeds=[0, 1, 2],                 # independent patient-fold assignments (stability of every headline number)
    primary_endpoint=dict(harm='fn', alpha=0.03, contrast='Full gate vs binary baseline'),
    ask_grid_a=[0.05, 0.10, 0.20, 0.30, 0.50],
    ask_grid_f=[0.0, 0.10, 0.25],
    lam_grid=np.geomspace(1e-5, 1.0, 240).tolist(),
    stress_seed=7,
    # regularisation of the cheap linear heads (explored on tuning patients only; see notebook text)
    crit_labels=['Edema', 'Consolidation', 'Pneumonia', 'Pneumothorax'],   # critical findings (severity >= 0.7) with their own risk guarantee
    crit_fracs=[0.75, 0.5],          # per-label target = fraction x that label's answer-everything miss rate
    ensemble_signal=True, ensemble_seeds=[0], ens_B=3,        # bootstrap-ensemble disagreement signal E (costly: run on seed 0 only)
    gate_train='nested',                 # 'dev' = gate models trained on tuning patients only; 'nested' = + out-of-fold predictions of fit patients, bagged
    head_C=dict(img=.01, ic=.001, il=.002, icl=.001, txt=.03),
    # deployment scenarios for the *clinical-context* information request (ASK-CONTEXT)
    schemes=['natural', 'mcar30', 'mcar60', 'struct40'],
    train_scheme='mcar30', primary_scheme='mcar30',
    ctx_a=0.10, ctx_f=0.05,                 # assumed cost / failure of asking the referrer for history
    ctx_grid_a=[0.02, 0.05, 0.10, 0.20, 0.30], ctx_grid_f=[0.0, 0.05, 0.20],
    lat_grid_a=[0.10, 0.25, 0.50],
)

def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def json_write(path, obj):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, default=str, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def atomic_npz(path, **arrays):
    path = Path(path); temp = path.with_name(path.name + '.tmp')
    with temp.open('wb') as h:
        np.savez_compressed(h, **arrays)
    temp.replace(path)

def read_npz(path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

def safe_prob(p):
    return np.clip(np.asarray(p, float), EPS, 1-EPS)

def check_disjoint(data):
    for i, a in enumerate(PARTS):
        assert len(data[a]['patient']) > 0, f'Empty partition: {a}'
        for b in PARTS[i+1:]:
            assert not set(data[a]['patient']) & set(data[b]['patient']), f'Patient overlap: {a}/{b}'
    for d in data.values():
        n = len(d['patient'])
        assert d['y'].shape == d['mask'].shape == (n, len(LABELS))
        assert np.isfinite(d['image']).all() and np.isfinite(d['text']).all() and np.isfinite(d['lateral']).all()
        assert np.isin(d['y'], [0, 1]).all() and np.isin(d['mask'], [0, 1]).all()

# ------------------------------------------------------------------ outcomes
def outcome(p, y, mask):
    """Per-study outcome of answering autonomously at the fixed 0.5 cutoff (identical to V4-V13)."""
    fn = (y > .5) & (mask > .5) & (p < CUTOFF)
    fp = (y < .5) & (mask > .5) & (p >= CUTOFF)
    return dict(fn=(fn*SEVERITY).sum(1)/SEVERITY.sum(),
                severe=(fn[:, SEVERE].any(1)).astype(float),
                fp=fp.sum(1)/np.maximum((mask > .5).sum(1), 1))

# ------------------------------------------------------------------ classifier heads (numeric, cheap)
class Heads:
    """One standardiser + one regularised logistic regression per label."""
    def __init__(self, C=.1):
        self.C = C
    def fit(self, x, y, mask):
        self.scaler = StandardScaler().fit(x)
        z = self.scaler.transform(x)
        self.models = []
        for j in range(y.shape[1]):
            keep = mask[:, j] > .5
            yy = y[keep, j]
            if len(yy) < 20 or min(yy.sum(), len(yy)-yy.sum()) < 3:
                self.models.append(float((yy.sum()+1)/(len(yy)+2)))
            else:
                m = LogisticRegression(C=self.C, max_iter=300)
                m.fit(z[keep], yy)
                self.models.append(m)
        return self
    def predict(self, x):
        z = self.scaler.transform(x)
        return safe_prob(np.column_stack([np.full(len(z), m) if isinstance(m, float)
                                          else m.predict_proba(z)[:, 1] for m in self.models]))

def fit_temperature(p, y, mask, min_each=30):
    grid = np.geomspace(.25, 8, 61)
    def nll(pp, yy):
        q = safe_prob(expit(logit(safe_prob(pp))[:, None]/grid))
        return -(yy[:, None]*np.log(q)+(1-yy[:, None])*np.log1p(-q)).mean(0)
    keep = mask > .5
    tg = float(grid[np.argmin(nll(p[keep], y[keep]))]) if keep.sum() > 50 else 1.
    out = []
    for j in range(p.shape[1]):
        k = mask[:, j] > .5; yy = y[k, j]
        if min(yy.sum(), len(yy)-yy.sum()) < min_each:
            out.append(tg)
        else:
            out.append(float(grid[np.argmin(nll(p[k, j], yy))]))
    return np.array(out)

def apply_temperature(p, t):
    return safe_prob(expit(logit(safe_prob(p))/t))

def mean_brier(p, y, mask):
    return float((((p-y)**2)*mask).sum()/max(mask.sum(), 1))

def blend(pi, pt, present, w):
    return np.where(present[:, None], (1-w)*pi+w*pt, pi)

# ------------------------------------------------------------------ conformal (Mondrian / class-conditional)
def conformal_q(p, y, mask, alpha):
    q = np.ones((p.shape[1], 2))
    for j in range(p.shape[1]):
        for cls in [0, 1]:
            k = (mask[:, j] > .5) & (y[:, j] == cls)
            score = p[k, j] if cls == 0 else 1-p[k, j]
            n = int(k.sum()); rank = int(np.ceil((n+1)*(1-alpha)))
            if n and rank <= n:
                q[j, cls] = np.sort(score)[rank-1]
    return q

def pred_sets(p, q):
    return np.stack([p <= q[:, 0], (1-p) <= q[:, 1]], axis=-1)

def wilson(successes, n, z=1.959963984540054):
    if n == 0:
        return np.nan, np.nan
    p = successes/n; den = 1+z*z/n
    mid = (p+z*z/(2*n))/den
    half = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return float(mid-half), float(mid+half)

# ------------------------------------------------------------------ the five signals
def kl(a, b):
    return a*np.log(a/b)+(1-a)*np.log((1-a)/(1-b))

def signal_blocks(p0, pi, pt, present, img, txt, aux, q):
    """U uncertainty | D cross-modal disagreement | M missing context | S clinical severity.
    Only information available BEFORE any extra data is requested. Workload (W) acts on the threshold, not here."""
    s = pred_sets(p0, q); amb = s.sum(2) == 2
    ent = -(p0*np.log(p0)+(1-p0)*np.log1p(-p0))
    U = np.column_stack([p0, ent.mean(1), amb.mean(1), (s.sum(2) == 0).mean(1), np.abs(p0-.5).min(1)])
    mid = (pi+pt)/2
    js = .5*(kl(pi, mid)+kl(pt, mid))
    pr = present.astype(float)
    D = np.column_stack([js.mean(1)*pr, js.max(1)*pr, ((pi >= .5) != (pt >= .5)).mean(1)*pr, pr])
    zi = aux['img_scaler'].transform(img)
    atyp_i = ((zi-aux['img_pca'].inverse_transform(aux['img_pca'].transform(zi)))**2).mean(1)
    zt = aux['txt_scaler'].transform(txt)
    atyp_t = ((zt-aux['txt_pca'].inverse_transform(aux['txt_pca'].transform(zt)))**2).mean(1)*pr
    M = np.column_stack([1-pr, atyp_i, atyp_t])
    S = np.column_stack([(p0*SEVERITY).sum(1)/SEVERITY.sum(), (amb*SEVERITY).max(1),
                         (p0*SEVERITY*(p0 < .5)).sum(1)/SEVERITY.sum(), (p0*SEVERITY).max(1)])
    return dict(U=U, D=D, M=M, S=S)

GATE_SETS = {'U': ['U'], 'UD': ['U', 'D'], 'UDM': ['U', 'D', 'M'], 'UDMS': ['U', 'D', 'M', 'S']}
def design(blocks, groups):
    return np.concatenate([blocks[g] for g in groups], axis=1)

def hgb(seed, n_iter=80):
    return HistGradientBoostingRegressor(max_iter=n_iter, max_leaf_nodes=7, min_samples_leaf=20,
                                         l2_regularization=2., learning_rate=.08, random_state=seed)

def patient_weights(patients):
    _, inv, counts = np.unique(patients, return_inverse=True, return_counts=True)
    w = 1/counts[inv]
    return w/w.mean()

# ------------------------------------------------------------------ guarantees: CRC and Learn-then-Test (Hoeffding-Bentkus)
def hb_pvalue(risk_hat, n, alpha):
    """Valid p-value for H0: risk > alpha, for losses bounded in [0,1] (Angelopoulos et al., Learn-then-Test)."""
    r = np.minimum(risk_hat, alpha)
    h1 = xlogy(r, r/alpha)+xlogy(1-r, (1-r)/(1-alpha))
    hoeffding = np.exp(-n*h1)
    bentkus = np.e*binom.cdf(np.ceil(n*risk_hat-1e-12), n, alpha)
    return np.minimum(1., np.minimum(hoeffding, bentkus))

def ltt_index(L, alpha, delta):
    """L: n x G calibration losses, columns ordered conservative -> aggressive. Fixed-sequence testing.
    Returns the last column of the unbroken run of rejections, or -1 if even the first fails."""
    n = L.shape[0]
    ok = hb_pvalue(L.mean(0), n, alpha) <= delta
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else L.shape[1]-1

def crc_index(L, alpha, B=1.0):
    """Conformal risk control for monotone families: largest aggressive column with (n*R+B)/(n+1) <= alpha."""
    n = L.shape[0]
    ok = (L.sum(0)+B)/(n+1) <= alpha
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else L.shape[1]-1

ESC_RATES = np.linspace(1, 0, 201)          # score gates: escalation fraction grid, conservative -> aggressive

def gate_thresholds(score_cal):
    t = np.quantile(score_cal, 1-ESC_RATES)
    t[0] = -np.inf
    return t

def gate_loss_matrix(score, harm, thresholds):
    return harm[:, None]*(score[:, None] < thresholds[None, :])

def calibrate_gate(score_cal, harm_cal, alpha, delta, method='ltt'):
    t = gate_thresholds(score_cal)
    L = gate_loss_matrix(score_cal, harm_cal, t)
    idx = ltt_index(L, alpha, delta) if method == 'ltt' else crc_index(L, alpha)
    return (-np.inf if idx < 0 else float(t[idx])), idx

# ------------------------------------------------------------------ 3-action routing (answer / escalate / ask)
def route_lagrange(c0hat, gainhat, eligible, lam, e, f, a, allow_ask):
    c0 = np.maximum(c0hat, 0.)
    c1 = np.maximum(c0hat-gainhat, 0.)
    obj = np.stack([c0, e*c0+lam,
                    np.where(allow_ask & eligible, (1-f)*c1+f*e*c0+lam*(a+f), np.inf)], axis=1)
    return obj.argmin(1)                    # 0 answer, 1 escalate, 2 ask

def realised(act, c0, c1, e, f):
    return np.where(act == 0, c0, np.where(act == 1, e*c0, (1-f)*c1+f*e*c0))

def workload_of(act, a, f):
    return np.where(act == 1, 1., np.where(act == 2, a+f, 0.))

def pick_lambda(c0hat, gainhat, eligible, e, f, a, allow_ask, budget, lam_grid):
    """Smallest price of workload whose label-free workload fits the budget."""
    for lam in lam_grid:
        act = route_lagrange(c0hat, gainhat, eligible, lam, e, f, a, allow_ask)
        if workload_of(act, a, f).mean() <= budget+1e-9:
            return lam, act
    return lam_grid[-1], act

def topk_escalate(score, budget):
    n = len(score); k = int(np.floor(budget*n+1e-9))
    act = np.zeros(n, int)
    if k > 0:
        act[np.argsort(-score, kind='stable')[:k]] = 1
    return act

# ------------------------------------------------------------------ statistics
def boot_ci(x, reps=2000, seed=0):
    """Percentile 95% interval of the mean, a two-sided bootstrap p-value and the per-unit sd (for power calculations)."""
    x = np.asarray(x, float); n = len(x); rng = np.random.default_rng(seed)
    vals = np.empty(reps)
    for s in range(0, reps, 200):
        m = min(200, reps-s)
        vals[s:s+m] = x[rng.integers(0, n, (m, n))].mean(1)
    lo, hi = np.quantile(vals, [.025, .975])
    p = min(1., 2*(min((vals <= 0).sum(), (vals >= 0).sum())+1)/(reps+1))
    sd = float(x.std(ddof=1)) if n > 1 else float('nan')
    return dict(mean=float(x.mean()), lo=float(lo), hi=float(hi), n=n, p=float(p), sd=sd, mde80=float(2.8*sd/np.sqrt(n)) if n > 1 else float('nan'))

def holm(pvals):
    """Holm-Bonferroni adjusted p-values (family-wise error control)."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.
    for rank, i in enumerate(order):
        run = max(run, min(1., (m-rank)*p[i])); adj[i] = run
    return adj

def verdict(ci, lower_is_better=True):
    if ci['hi'] < 0:
        return 'better' if lower_is_better else 'worse'
    if ci['lo'] > 0:
        return 'worse' if lower_is_better else 'better'
    return 'inconclusive'

def auc_delta_ci(y, a, b, reps=1000, seed=0):
    y = np.asarray(y); n = len(y); rng = np.random.default_rng(seed); vals = []
    for _ in range(reps):
        i = rng.integers(0, n, n)
        if len(np.unique(y[i])) < 2:
            continue
        vals.append([roc_auc_score(y[i], b[i])-roc_auc_score(y[i], a[i]),
                     average_precision_score(y[i], b[i])-average_precision_score(y[i], a[i])])
    vals = np.asarray(vals)
    out = {}
    for j, (name, fn) in enumerate([('auroc', roc_auc_score), ('auprc', average_precision_score)]):
        lo, hi = np.quantile(vals[:, j], [.025, .975])
        out[name] = dict(a=float(fn(y, a)), b=float(fn(y, b)), delta=float(fn(y, b)-fn(y, a)), lo=float(lo), hi=float(hi))
    return out


# ------------------------------------------------------------------ four-action routing: answer / escalate / ask-context / ask-lateral
def route4(c0hat, gc, gl, ctx_ok, lat_ok, lam, e, p):
    c0 = np.maximum(c0hat, 0.)
    cc = np.maximum(c0-gc, 0.); cl = np.maximum(c0-gl, 0.)
    obj = np.stack([c0, e*c0+lam,
                    np.where(p['allow_ctx'] & ctx_ok, (1-p['f_c'])*cc+p['f_c']*e*c0+lam*(p['a_c']+p['f_c']), np.inf),
                    np.where(p['allow_lat'] & lat_ok, (1-p['f_l'])*cl+p['f_l']*e*c0+lam*(p['a_l']+p['f_l']), np.inf)], axis=1)
    return obj.argmin(1)                    # 0 answer, 1 escalate, 2 ask context, 3 ask lateral

def realised4(act, c0, cc, cl, e, p):
    return np.select([act == 0, act == 1, act == 2, act == 3],
                     [c0, e*c0, (1-p['f_c'])*cc+p['f_c']*e*c0, (1-p['f_l'])*cl+p['f_l']*e*c0])

def workload4(act, p):
    return np.select([act == 1, act == 2, act == 3], [1., p['a_c']+p['f_c'], p['a_l']+p['f_l']], 0.)

def pick_lambda4(c0hat, gc, gl, ctx_ok, lat_ok, e, p, budget, lam_grid):
    for lam in lam_grid:
        act = route4(c0hat, gc, gl, ctx_ok, lat_ok, lam, e, p)
        if workload4(act, p).mean() <= budget+1e-9:
            return lam, act
    return lam_grid[-1], act

def hash_uniform(strings, salt):
    return np.array([int(hashlib.sha256(f'{salt}:{s}'.encode()).hexdigest()[:12], 16)/16**12 for s in strings])


# ------------------------------------------------------------------ learning-to-defer baseline (surrogate-loss gate)
from scipy.optimize import minimize
class L2DGate:
    """Learning-to-defer baseline in the spirit of Madras et al. / Mozannar & Sontag: a gate g(x) trained to minimise the expected team cost
    E[(1-g)*c + g*(e*c + lam)] for a fixed classifier and an expert that leaves a residual e of the loss. It is trained directly on the
    deferral objective (no intermediate loss regression). To give it a fair function class it uses all degree-2 feature interactions."""
    def fit(self, X, c, e, lam, w=None, reg=3e-3):
        from sklearn.preprocessing import PolynomialFeatures
        self.poly = PolynomialFeatures(2, include_bias=False)
        Z0 = self.poly.fit_transform((X-X.mean(0))/(X.std(0)+1e-8)); self.mu0, self.sd0 = X.mean(0), X.std(0)+1e-8
        self.mu = Z0.mean(0); self.sd = Z0.std(0)+1e-8; Z = (Z0-self.mu)/self.sd
        w = np.ones(len(X)) if w is None else w/np.mean(w)
        delta = (e*c+lam)-c
        def f(theta):
            g = expit(Z@theta[:-1]+theta[-1])
            loss = np.mean(w*(c+g*delta))+reg*np.sum(theta[:-1]**2)
            dz = w*g*(1-g)*delta/len(Z)
            return loss, np.concatenate([Z.T@dz+2*reg*theta[:-1], [dz.sum()]])
        self.theta = minimize(f, np.zeros(Z.shape[1]+1), jac=True, method='L-BFGS-B', options=dict(maxiter=400)).x
        return self
    def score(self, X):
        Z = (self.poly.transform((X-self.mu0)/self.sd0)-self.mu)/self.sd
        return Z@self.theta[:-1]+self.theta[-1]


def multi_ltt_index(Ls, alphas, delta):
    """Fixed-sequence Learn-then-Test for SEVERAL risks at once: a column is accepted only if every risk's Hoeffding-Bentkus p-value is <= delta
    (delta already includes the Bonferroni split across risks). Returns the last column of the unbroken accepted run, or -1."""
    n = Ls[0].shape[0]
    ok = np.ones(Ls[0].shape[1], bool)
    for L, a in zip(Ls, alphas):
        ok &= hb_pvalue(L.mean(0), n, a) <= delta
    if not ok[0]:
        return -1
    bad = np.flatnonzero(~ok)
    return int(bad[0]-1) if len(bad) else Ls[0].shape[1]-1

## 1. Load the V19 checkpoints
Reads the out-of-sample arrays and calibration sets. The class stubs below are needed to unpickle the saved gate models.

In [ ]:
K = PLAN['folds']
_cands = sorted((RUN_DIR/'methodology_v19').glob('*/seed0/fold0.npz'), key=lambda p: p.stat().st_mtime)
if not _cands:
    raise FileNotFoundError('No V19 fold checkpoints found under '+str(RUN_DIR/'methodology_v19')+'. Run the main V19 notebook first (it writes seed0/fold*.npz).')
OUT = _cands[-1].parent
print('Using fold checkpoints from', OUT)
BASE_KEYS = ['case', 'patient', 'eligible', 'present', 'y', 'mask', 'pr_img', 'pr_ic', 'pr_il', 'pr_icl']
SCHEME_KEYS = ['avail', 'ctx_ok', 'lat_ok', 'p0', 'dmean', 'fn0', 'fp0', 'sev0', 'fnc', 'fpc', 'sevc', 'fnl', 'fpl', 'sevl',
               'gc', 'gl', 'sc_entropy', 'sc_ambiguity', 'sc_sevharm', 'sc_l2d_U', 'sc_l2d_UDMS'] +               [f'{a}_{F}' for F in GATE_SETS for a in ['sc_fn', 'sc_sv', 'c0hat']] + \
              [f'miss_{l}' for l in PLAN['crit_labels']] + [f'scj_{l}_{F}' for l in PLAN['crit_labels'] for F in ['U', 'UDMS']]
def pooled(role, key):
    return np.concatenate([F[f'{role}_{key}'] for F in FOLDS])
def load_seed():
    """(Re)bind the pooled out-of-sample arrays of the CURRENT seed's folds (global OUT)."""
    global FOLDS, EVB, NE, FS
    FOLDS = [read_npz(OUT/f'fold{k}.npz') for k in range(K)]
    EVB = {key: pooled('ev', key) for key in BASE_KEYS}
    EVB['fold'] = np.concatenate([np.full(len(F['ev_case']), k) for k, F in enumerate(FOLDS)])
    NE = len(EVB['case'])
    assert len(set(EVB['patient'])) == NE, 'each evaluation patient must appear exactly once'
    FS = [np.flatnonzero(EVB['fold'] == k) for k in range(K)]
load_seed()
def build_EV(scheme):
    ev = dict(EVB)
    ev.update({key: pooled('ev', f'{scheme}_{key}') for key in SCHEME_KEYS})
    return ev
def get_cal(k, scheme):
    F = FOLDS[k]
    return {key: F[f'cal_{scheme}_{key}'] for key in SCHEME_KEYS}
PRIMARY = PLAN['primary_scheme']
EV = build_EV(PRIMARY)
RESULTS = {}      # everything that ends up in the report
TABLES = {}       # DataFrames exported to Excel/CSV
FPW = PLAN['fp_weight']
print(f'{NE:,} evaluation patients loaded from the checkpoints.')


class Bag:                                   # same definitions as V19 (needed to unpickle the fold bundles)
    def __init__(self, members):
        self.members = members
    def predict(self, X):
        return np.mean([m.predict(X) for m in self.members], axis=0)

class _Zero:
    def predict(self, X):
        return np.zeros(len(X))

frozen = json.loads((RUN_DIR/'frozen_manifest.json').read_text()); CFG = dict(frozen['config'])
original_cfg = json.loads((RUN_DIR/'config.json').read_text())
encoder = json.loads((RUN_DIR/'encoder_manifest.json').read_text())
ASK_A = float(CFG.get('ask_workload', 0.25)); ASK_F = float(CFG.get('retrieval_failure', 0.15)); EXPERT_E = float(CFG.get('expert_residual', 0.10))
P_ASK = dict(a_c=PLAN['ctx_a'], f_c=PLAN['ctx_f'], a_l=ASK_A, f_l=ASK_F)
LAM = np.array(PLAN['lam_grid'])
for k in range(K):
    assert (OUT/f'fold{k}_bundle_PRIVATE.pkl').is_file(), f'missing fold{k}_bundle_PRIVATE.pkl in {OUT}'
print('Encoder manifest:', encoder.get('image_model'), '|', encoder.get('text'), '| max_tokens', encoder.get('max_tokens'), '|', encoder.get('pooling'))
print(f'Assumed: lateral request a={ASK_A}, f={ASK_F}; context request a={P_ASK["a_c"]}, f={P_ASK["f_c"]}; expert residual e={EXPERT_E}')

## 2. Connect to CheXpert Plus and inspect the schema
This cell reads only the catalogue (table names, variable names and row counts), not patient records or images. The first Redivis call prints a sign-in link: open it, sign in with the account that has CheXpert Plus access, then return to the notebook.

In [ ]:
# ---------------------------------------------------------------- CheXpert Plus (Stanford AIMI, via Redivis): connect + schema discovery
I_CONFIRM_CHEXPERT_PLUS_TERMS = False     # set True only after checking the Stanford AIMI research use agreement (gates every row/image read below)
CXP_ORG, CXP_DATASET = 'AIMI', 'chexpert_plus:5yyj:v1_0'
CXP_DIR = RUN_DIR/'external_chexpert_plus'; CXP_DIR.mkdir(exist_ok=True)
if importlib.util.find_spec('redivis') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'redivis'])
import redivis
CXP = redivis.organization(CXP_ORG).dataset(CXP_DATASET)
CXP.get()
_props = dict(CXP.properties or {})
print('Dataset:', _props.get('name'), '| version', (_props.get('version') or {}).get('tag'), '| tables', _props.get('tableCount'),
      '| rows', _props.get('totalNumRows'), '| bytes', _props.get('totalNumBytes'))

def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

SCHEMA = {}
for t in CXP.list_tables():
    t.get(); tp = dict(t.properties or {})
    vars_ = [v.name for v in t.list_variables()]
    is_files = bool(tp.get('isFileIndex') or tp.get('fileCount')) or any(norm(v) == 'fileid' for v in vars_)
    SCHEMA[t.name] = dict(rows=tp.get('numRows'), files=tp.get('fileCount'), file_index=is_files, variables=vars_, table=t)
    print(f"\n[{t.name}]  rows={tp.get('numRows')}  files={tp.get('fileCount')}  file_index={is_files}")
    print('   ', ', '.join(vars_[:90]) + (' ...' if len(vars_) > 90 else ''))
json_write(CXP_DIR/'schema_summary.json', {k: {kk: vv for kk, vv in v.items() if kk != 'table'} for k, v in SCHEMA.items()})
print('\nSchema summary saved (no patient data):', CXP_DIR/'schema_summary.json')

## 3. Map the schema onto the roles the analysis needs
Automatic first. **Read the printed mapping.** Wherever it shows `None` or the wrong table or variable, set it in `OVERRIDE` using the exact name printed in section 2, then re-run this cell.

In [ ]:
# ---------------------------------------------------------------- schema mapping (automatic, with explicit overrides)
OVERRIDE = dict(meta_table=None, label_table=None, second_label_table=None, file_tables=None,      # file_tables: list of table names
                path=None, patient=None, view=None, ap_pa=None, report=None, age=None, sex=None, race=None, ethnicity=None)
LAB_N = {norm(l): l for l in LABELS}
LAB_N.update({'airspaceopacity': 'Lung Opacity', 'supportdevice': 'Support Devices', 'fractures': 'Fracture', 'pleuralothers': 'Pleural Other'})
SECTION_NAMES = {'indication': 'INDICATION', 'clinicalindication': 'CLINICAL INDICATION', 'reasonforexam': 'REASON FOR EXAM',
                 'reasonforexamination': 'REASON FOR EXAMINATION', 'history': 'HISTORY', 'clinicalhistory': 'CLINICAL HISTORY', 'comparison': 'COMPARISON'}

def label_cols(vars_):
    out = {}
    for v in vars_:
        l = LAB_N.get(norm(v))
        if l and l not in out:
            out[l] = v
    return out

def find(vars_, exact, contains=()):
    nv = {norm(v): v for v in vars_}
    for e in exact:
        if norm(e) in nv:
            return nv[norm(e)]
    for c in contains:
        hits = [v for v in vars_ if c in norm(v)]
        if len(hits) == 1:
            return hits[0]
    return None

def path_col(vars_):
    return find(vars_, ['path_to_image', 'image_path', 'path', 'filepath'], ['pathtoimage'])

def meta_roles(vars_):
    r = dict(path=path_col(vars_),
             patient=find(vars_, ['deid_patient_id', 'patient_id', 'patient', 'subject_id'], ['patientid']),
             view=find(vars_, ['frontal_lateral', 'frontallateral', 'view'], ['frontallateral']),
             ap_pa=find(vars_, ['ap_pa', 'appa', 'view_position', 'viewposition'], ['appa']),
             report=find(vars_, ['report', 'report_text', 'full_report'], []),
             age=find(vars_, ['age', 'patient_age', 'age_at_study'], []),
             sex=find(vars_, ['sex', 'gender', 'patient_sex'], []),
             race=find(vars_, ['race'], []), ethnicity=find(vars_, ['ethnicity'], []))
    r['sections'] = {SECTION_NAMES[norm(v)[7:]]: v for v in vars_ if norm(v).startswith('section') and norm(v)[7:] in SECTION_NAMES}
    return r

def meta_score(name):
    r = meta_roles(SCHEMA[name]['variables'])
    return (not SCHEMA[name]['file_index'])*10 + 3*bool(r['path']) + 2*bool(r['view']) + 2*bool(r['report'] or r['sections']) + bool(r['age']) + bool(r['sex'])

M = {}
M['meta_table'] = OVERRIDE['meta_table'] or max(SCHEMA, key=meta_score)
M.update(meta_roles(SCHEMA[M['meta_table']]['variables']))
for key in ['path', 'patient', 'view', 'ap_pa', 'report', 'age', 'sex', 'race', 'ethnicity']:
    if OVERRIDE[key]:
        M[key] = OVERRIDE[key]
lab_tables = {n: label_cols(v['variables']) for n, v in SCHEMA.items() if len(label_cols(v['variables'])) >= 10 and path_col(v['variables'])}
def lab_rank(n):
    nn = n.lower()
    return 2*('chexpert' in nn and 'chexbert' not in nn) + ('impression' in nn) - 2*('chexbert' in nn) - ('finding' in nn and 'impression' not in nn)
M['label_table'] = OVERRIDE['label_table'] or (max(lab_tables, key=lab_rank) if lab_tables else None)
_second = [n for n in lab_tables if n != M['label_table'] and 'chexbert' in n.lower()]
M['second_label_table'] = OVERRIDE['second_label_table'] or (sorted(_second, key=lambda n: -('impression' in n.lower()))[0] if _second else None)
# Labels shipped as JSON files (CheXpert Plus v1.0: <section>_fixed.json in a small file-index table; listing file names only)
LABEL_FILES = {}
for n, v in SCHEMA.items():
    if v['file_index'] and int(v['rows'] or 0) <= 50:
        try:
            fl = v['table'].to_pandas_dataframe(max_results=50)
        except Exception as exc:
            print('  could not list files of', n, type(exc).__name__); continue
        for fid, fn in zip(fl['file_id'], fl['file_name']):
            if str(fn).endswith('_fixed.json'):
                LABEL_FILES[str(fn)[:-len('_fixed.json')]] = (n, fid, str(fn))
if not M['label_table'] and 'impression' in LABEL_FILES:
    M['label_table'] = 'files:impression'                 # impression labels, findings labels where a report has no impression (MIMIC labeler rule)
# report_fixed.json is NOT used: in v1.0 it lists the same paths in the same row order as impression_fixed.json, yet agreement with
# the impression labels is at chance level (kappa 0.00-0.05 over all 223,462 images), while impression labels track the images
# (transfer AUROC up to 0.88). Its labels appear misaligned with the paths, so no label-source check is run on CheXpert Plus.
# To inspect it anyway, set OVERRIDE['second_label_table'] = 'files:report'.
_ft = [n for n, v in SCHEMA.items() if v['file_index'] and 'compressed' not in n.lower() and int(v['rows'] or 0) > 50]
_pref = [n for n in _ft if 'png' in n.lower()] or [n for n in _ft if 'jpg' in n.lower() or 'jpeg' in n.lower()] or [n for n in _ft if 'dcm' not in n.lower() and 'dicom' not in n.lower()] or _ft
M['file_tables'] = OVERRIDE['file_tables'] or _pref
print('Label-table candidates (>=10 of the 14 labels):', {n: len(c) for n, c in lab_tables.items()})
print('Label JSON files:', {k_: v_[2] for k_, v_ in LABEL_FILES.items()})
print('Image file-index tables:', _ft)
print('\nMAPPING')
for k_, v_ in M.items():
    print(f'  {k_:20s} {v_}')
problems = [k_ for k_ in ['meta_table', 'path', 'label_table'] if not M[k_]] + ([] if M['file_tables'] else ['file_tables'])
if not (M['report'] or M['sections']):
    problems.append('report or section_* columns (clinical history)')
if problems:
    print('\nUNRESOLVED:', problems, '-> set them in OVERRIDE and re-run this cell. Label JSON files are detected automatically (<section>_fixed.json).')
else:
    print('\nMapping complete. Context source:', 'pre-split section columns' if M['sections'] else 'full report parsed with the MIMIC section parser')

## 4. Build the external cohort (one study per patient, chosen by hash)
Uses the same rules as MIMIC:
- **Frontal image:** PA preferred over AP.
- **Lateral:** the first lateral view of the same study.
- **Context:** only the history, indication and comparison sections; findings and impression are never used.
- **Labels:** uncertain values masked, blank counted as negative.

The cohort is saved to Drive as a PRIVATE file.

In [ ]:
# ---------------------------------------------------------------- external cohort (label-blind hashed selection, N_CXP patients)
assert I_CONFIRM_CHEXPERT_PLUS_TERMS, 'Set I_CONFIRM_CHEXPERT_PLUS_TERMS = True in section 2 after checking the data use agreement.'
N_CXP = 6000                                   # same size as the NIH cohort
SELECTION_SEED = 'ARG-V20-chexpert-plus'
import tempfile
COHORT_PKL = CXP_DIR/'cohort_PRIVATE.pkl'
ALLOWED_SECTIONS = {'INDICATION', 'HISTORY', 'CLINICAL HISTORY', 'CLINICAL INDICATION', 'REASON FOR EXAM', 'REASON FOR EXAMINATION', 'COMPARISON'}
HEADER = re.compile(r'(?mi)^[ \t]*([A-Z][A-Z /_-]{1,60}):[ \t]*')
FORBIDDEN_BOUNDARY = re.compile(r'(?im)\b(?:findings|impression|conclusion|interpretation)\s*:|^[ \t]*(?:findings|impression|conclusion|interpretation)[ \t]*$')

def context_only(report):                      # identical to the MIMIC parser
    boundary = FORBIDDEN_BOUNDARY.search(report)
    if boundary:
        report = report[:boundary.start()]
    headings = list(HEADER.finditer(report)); chunks = []
    for k_, h in enumerate(headings):
        end = headings[k_+1].start() if k_+1 < len(headings) else len(report)
        if h.group(1).strip().upper() in ALLOWED_SECTIONS:
            chunks.append(h.group(1).strip().upper()+': '+report[h.end():end].strip())
    return '\n'.join(chunks)

def img_key(p):
    """'.../patient00001/study1/view1_frontal.jpg' -> 'patient00001/study1/view1_frontal' (extension- and root-independent)."""
    parts = Path(str(p).replace('\\', '/')).with_suffix('').parts
    return '/'.join(parts[-3:]).lower()

def clean(s):
    s = '' if s is None or (np.isscalar(s) and pd.isna(s)) or s is pd.NA else str(s).strip()
    return '' if s.lower() in ('nan', 'none', 'null', '<na>') else s

def race_group(r, e):
    r, e = clean(r).lower(), clean(e).lower()
    if 'hispanic' in e and 'non' not in e and 'not' not in e:
        return 'Hispanic'
    if 'white' in r:
        return 'White'
    if 'black' in r or 'african' in r:
        return 'Black'
    if 'asian' in r:
        return 'Asian'
    return 'Other/unknown'

def read_file_bytes(table_name, file_id):
    t = SCHEMA[table_name]['table']; errs = []
    for get in (lambda: t.file(file_id).read(), lambda: redivis.File(file_id).read()):
        try:
            return get()
        except Exception as exc:
            errs.append(f'{type(exc).__name__}: {exc}')
    try:
        with tempfile.TemporaryDirectory() as td:
            got = t.file(file_id).download(path=td, overwrite=True)
            return Path(got).read_bytes()
    except Exception as exc:
        errs.append(f'download: {type(exc).__name__}: {exc}')
    raise RuntimeError(' | '.join(errs))

def labels_frame(df):
    vars_ = list(df.columns); lc = label_cols(vars_); kcol = path_col(vars_)
    if kcol is None:
        raise RuntimeError('label data has no image-path column; columns: '+str(vars_[:30]))
    out = pd.DataFrame({'key': df[kcol].map(img_key)})
    for lab in LABELS:
        out[lab] = pd.to_numeric(df[lc[lab]], errors='coerce').to_numpy() if lab in lc else np.nan
    out['_found'] = 1.0
    return out.drop_duplicates('key'), [l for l in LABELS if l not in lc]

def read_labels(src):
    if not src.startswith('files:'):
        vars_ = SCHEMA[src]['variables']
        return labels_frame(SCHEMA[src]['table'].to_pandas_dataframe(variables=[path_col(vars_)]+list(label_cols(vars_).values())))
    tn, fid, fn = LABEL_FILES[src[len('files:'):]]
    raw = read_file_bytes(tn, fid).decode('utf-8')
    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:                   # JSON lines
        obj = [json.loads(l) for l in raw.splitlines() if l.strip()]
    df = None
    if isinstance(obj, dict):
        try:
            df = pd.DataFrame(obj)
        except ValueError:
            df = None
        if df is None or path_col(list(df.columns)) is None:     # {path: {label: value}}
            df = pd.DataFrame.from_dict(obj, orient='index').rename_axis('path_to_image').reset_index()
    else:
        df = pd.DataFrame(obj)
    lab, miss = labels_frame(df)
    vals = pd.Series(lab[LABELS].to_numpy().ravel()).value_counts(dropna=False).head(6).to_dict()
    print(f'  {fn}: {len(lab):,} images, columns {list(df.columns)[:6]}..., label values {vals}')
    return lab, miss

def label_arrays(lab, missing, keys):
    A = lab.set_index('key').reindex(keys)
    found = A['_found'].eq(1).to_numpy()
    if (~found).any():                         # labels may be stored once per study: fall back to the study of the image
        bys = lab.assign(study=lab.key.str.rsplit('/', n=1).str[0]).drop_duplicates('study').set_index('study')
        B = bys.reindex([k_.rsplit('/', 1)[0] for k_ in np.asarray(keys)[~found]])
        A.iloc[np.flatnonzero(~found)] = B[A.columns].to_numpy()
        found = A['_found'].eq(1).to_numpy()
    raw = A[LABELS].to_numpy(float)
    mask = (raw != -1).astype(float)           # MIMIC rule: uncertain masked, blank = negative
    for l in missing:
        mask[:, LABELS.index(l)] = 0.          # a label the table does not have is unobserved, not negative
    y = (np.nan_to_num(raw, nan=0) > 0).astype(float)
    mask[~found] = 0.
    return y, mask, found

def build_cohort():
    T = lambda name: SCHEMA[name]['table']
    cols = [M[k_] for k_ in ['path', 'patient', 'view', 'ap_pa', 'age', 'sex', 'race', 'ethnicity'] if M[k_]] + list(M['sections'].values())
    if not M['sections']:
        cols.append(M['report'])
    mv = SCHEMA[M['meta_table']]['variables']
    imp_col, fnd_col = find(mv, ['section_impression'], []), find(mv, ['section_findings'], [])
    cols += [c for c in [imp_col, fnd_col] if c]               # only to know which section a report has (label source rule)
    meta = T(M['meta_table']).to_pandas_dataframe(variables=list(dict.fromkeys(cols)))
    print(f'Metadata rows: {len(meta):,}')
    meta['key'] = meta[M['path']].map(img_key)
    parts = meta['key'].str.split('/')
    meta['study'] = parts.str[0]+'/'+parts.str[1]
    meta['patient'] = meta[M['patient']].astype(str) if M['patient'] else parts.str[0]
    vtxt = meta[M['view']].astype(str).str.lower() if M['view'] else meta['key']
    meta['is_lat'] = vtxt.str.contains('lateral')
    meta['is_front'] = vtxt.str.contains('frontal') & ~meta['is_lat'] if M['view'] else ~meta['is_lat']
    appa = meta[M['ap_pa']].astype(str).str.upper().str.strip() if M['ap_pa'] else pd.Series('', index=meta.index)
    meta['prio'] = appa.map({'PA': 0, 'AP': 1}).fillna(2)
    front = meta[meta.is_front].sort_values(['patient', 'study', 'prio', 'key']).drop_duplicates(['patient', 'study']).copy()
    lat = meta[meta.is_lat].sort_values('key').drop_duplicates(['patient', 'study'])[['patient', 'study', 'key']].rename(columns={'key': 'lat_key'})
    front = front.merge(lat, on=['patient', 'study'], how='left', validate='one_to_one')
    print(f'Frontal studies: {len(front):,} from {front.patient.nunique():,} patients; with a lateral view: {front.lat_key.notna().mean():.1%}')
    # ---- context (history / indication / comparison only)
    if M['sections']:
        front['context'] = ['\n'.join(f'{nm}: {clean(r[c])}' for nm, c in M['sections'].items() if clean(r[c])) for _, r in front.iterrows()]
    else:
        rep = front[M['report']].fillna('').astype(str).map(lambda s: re.sub(r'\s+(?=[A-Z][A-Z /_-]{1,60}:)', '\n', s))
        front['context'] = rep.map(context_only)
    hit = front.context.map(lambda s: FORBIDDEN_BOUNDARY.search(s))
    leak = hit.notna().mean()
    front['context'] = [s[:h.start()].strip() if h else s for s, h in zip(front.context, hit)]   # cut at the boundary, as the MIMIC parser does
    assert not front.context.map(lambda s: bool(FORBIDDEN_BOUNDARY.search(s))).any()
    print(f'Context non-empty: {front.context.str.strip().ne("").mean():.1%}; contexts cut at a findings/impression header: {leak:.4%}')
    print('Context sections present:', {nm: float(front[c].map(clean).ne('').mean()) for nm, c in M['sections'].items()} if M['sections'] else 'parsed from report')
    # ---- labels
    keys = front.key.to_numpy()
    if M['label_table'] == 'files:impression':
        has_imp = front[imp_col].map(clean).ne('').to_numpy() if imp_col else np.ones(len(front), bool)
        has_fnd = front[fnd_col].map(clean).ne('').to_numpy() if fnd_col else np.zeros(len(front), bool)
        li, mi = read_labels('files:impression'); yi, mki, fi = label_arrays(li, mi, keys); fi &= has_imp
        if 'findings' in LABEL_FILES:
            lf_, mf_ = read_labels('files:findings'); yf, mkf, ff = label_arrays(lf_, mf_, keys); ff &= has_fnd & ~fi
        else:
            yf, mkf, ff = yi, mki, np.zeros(len(front), bool)
        y = np.where(fi[:, None], yi, yf); mask = np.where(fi[:, None], mki, mkf); found = fi | ff; mask[~found] = 0.
        miss = sorted(set(mi) | set(mf_ if 'findings' in LABEL_FILES else []))
        print(f'Label source: impression {fi.mean():.1%} of studies, findings (no impression) {ff.mean():.1%}, unlabeled {(~found).mean():.1%}')
    else:
        lab, miss = read_labels(M['label_table'])
        y, mask, found = label_arrays(lab, miss, keys)
    print('Labels from', M['label_table'], '| matched studies:', f'{found.mean():.1%}', '| labels absent:', miss)
    keep = found & (mask.sum(1) > 0)
    front, y, mask = front[keep].reset_index(drop=True), y[keep], mask[keep]
    # ---- one study per patient (label-blind hash), then N_CXP patients by hash
    front['rank'] = [stable_hash([SELECTION_SEED, p, s]) for p, s in zip(front.patient, front.study)]
    pick = front.sort_values(['patient', 'rank']).drop_duplicates('patient')
    pick = pick.assign(prank=[stable_hash([SELECTION_SEED, 'patient', p]) for p in pick.patient]).sort_values('prank')
    # ---- image files
    fr = []
    for tn in M['file_tables']:
        fv = SCHEMA[tn]['variables']
        idc = find(fv, ['file_id'], ['fileid']); nmc = find(fv, ['file_name', 'filename', 'name'], ['filename'])
        f = T(tn).to_pandas_dataframe(variables=[idc, nmc]); f.columns = ['file_id', 'file_name']; f['file_table'] = tn
        fr.append(f)
    files = pd.concat(fr, ignore_index=True)
    files['key'] = files.file_name.map(img_key)
    files['ext_rank'] = [{'.png': 0, '.jpg': 1, '.jpeg': 1, '.dcm': 2}.get(os.path.splitext(str(f).lower())[1], 3) for f in files.file_name]
    print('Example file names:', files.file_name.head(3).tolist())
    files = files.sort_values(['key', 'ext_rank']).drop_duplicates('key').set_index('key')
    pick = pick[pick.key.isin(files.index)]
    print(f'Frontal images matched to a Redivis file: {len(pick):,} patients')
    if len(pick) < 1000:
        raise RuntimeError('Too few images matched by path. Check file names above vs the path variable '+str(M['path'])+' (keys use the last 3 path parts).')
    pick = pick.head(N_CXP).reset_index(drop=True)
    pick['file_id'] = files.file_id.reindex(pick.key).to_numpy(); pick['file_table'] = files.file_table.reindex(pick.key).to_numpy()
    lk = pick.lat_key.where(pick.lat_key.isin(files.index))
    pick['lat_key'] = lk
    pick['lat_file_id'] = files.file_id.reindex(lk.fillna('-')).to_numpy(); pick['lat_file_table'] = files.file_table.reindex(lk.fillna('-')).to_numpy()
    idx = front.reset_index().set_index('key').loc[pick.key, 'index'].to_numpy()
    out = dict(frame=pick, y=y[idx], mask=mask[idx])
    if M['second_label_table']:
        lab2, miss2 = read_labels(M['second_label_table'])
        y2, m2, f2 = label_arrays(lab2, miss2, pick.key.to_numpy())
        out.update(y2=y2, mask2=m2); print('Alternative labels', M['second_label_table'], 'matched:', f'{f2.mean():.1%}')
    pick['sex_g'] = pick[M['sex']].map(lambda s: clean(s)[:1].upper()) if M['sex'] else ''
    pick['age_v'] = pd.to_numeric(pick[M['age']], errors='coerce') if M['age'] else np.nan
    pick['race_g'] = [race_group(r, e) for r, e in zip(pick[M['race']] if M['race'] else ['']*len(pick), pick[M['ethnicity']] if M['ethnicity'] else ['']*len(pick))]
    out['frame'] = pick
    pd.to_pickle(dict(out, mapping={k_: v_ for k_, v_ in M.items()}, n_target=N_CXP), COHORT_PKL)
    print(f'Saved cohort: {len(pick):,} patients | context present {pick.context.str.strip().ne("").mean():.1%} | lateral {pick.lat_key.notna().mean():.1%}')
    return out

COHORT = pd.read_pickle(COHORT_PKL) if COHORT_PKL.is_file() else None
if COHORT is None or COHORT.get('n_target') != N_CXP:          # a changed N_CXP rebuilds the cohort (hash order: the first patients stay the same)
    COHORT = build_cohort()
_f = COHORT['frame']
print('Cohort:', len(_f), 'patients | sex', _f.sex_g.value_counts().to_dict(), '| race', _f.race_g.value_counts().to_dict(),
      '| age median', float(np.nanmedian(_f.age_v)) if _f.age_v.notna().any() else 'n/a')
print('Positives per label:', dict(zip(LABELS, (COHORT['y']*COHORT['mask']).sum(0).astype(int))))

## 5. Encode with the same frozen encoders (resumable)
- **Image and lateral:** BiomedCLIP `encode_image` (512-d, not normalised), with the same transform and the commit pinned for the V13 and NIH runs.
- **Context:** BiomedBERT at the MIMIC revision, 192 tokens, mean pooling with the attention mask, 768-d.
- **Missing lateral or empty context:** stored as a zero vector, as in MIMIC.

The first three images are decoded as a test before the bulk run. Set the `CXP_IMAGE_ROOT` environment variable if you have your own copy of the CheXpert images; it is checked before Redivis.

In [ ]:
# ---------------------------------------------------------------- encoding (frontal, lateral, context); chunks checkpointed to Drive
CXP_NPZ = CXP_DIR/'cxp_embeddings_PRIVATE.npz'
PART_DIR = CXP_DIR/'enc_parts_PRIVATE'; PART_DIR.mkdir(exist_ok=True)
CHUNK, THREADS = 128, 8
IMAGE_ROOT = os.environ.get('CXP_IMAGE_ROOT')
import io, tempfile, concurrent.futures as cf

def load_encoders():
    for pkg, mod in [('open_clip_torch', 'open_clip'), ('huggingface_hub', 'huggingface_hub'), ('transformers', 'transformers'), ('timm', 'timm'), ('pydicom', 'pydicom')]:
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
    import torch, open_clip
    from transformers import AutoTokenizer, AutoModel
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    pin_file = RUN_DIR/V13_REL/'encoder_pin.json'
    pin = json.loads(pin_file.read_text()) if pin_file.is_file() else {}
    repo = 'microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
    try:
        from huggingface_hub import snapshot_download
        local = snapshot_download(repo, revision=pin['biomedclip_commit']) if pin.get('biomedclip_commit') else None
        model, _, preprocess = open_clip.create_model_and_transforms('local-dir:'+local if local else 'hf-hub:'+repo)
    except Exception as exc:
        print('Pinned-revision load failed ('+type(exc).__name__+'); using the default revision of', repo); pin['biomedclip_commit'] = None
        model, _, preprocess = open_clip.create_model_and_transforms('hf-hub:'+repo)
    tm, rev, maxtok = original_cfg['text_model'], original_cfg['text_revision'], int(original_cfg['max_tokens'])
    assert tm == encoder.get('text') and maxtok == int(encoder.get('max_tokens', maxtok)), 'text encoder settings differ from the MIMIC manifest'
    tok = AutoTokenizer.from_pretrained(tm, revision=rev); txt_model = AutoModel.from_pretrained(tm, revision=rev).to(dev).eval()
    model = model.to(dev).eval()
    if dev == 'cpu':
        torch.set_num_threads(os.cpu_count() or 2)
    print('Encoders on', dev, '| BiomedCLIP commit', pin.get('biomedclip_commit'), '| text', tm, rev[:8], '| max_tokens', maxtok)
    return torch, model, preprocess, tok, txt_model, maxtok, dev, pin.get('biomedclip_commit')

def decode(raw):
    from PIL import Image
    try:
        im = Image.open(io.BytesIO(raw)); im.load()
    except Exception:
        import pydicom
        ds = pydicom.dcmread(io.BytesIO(raw)); a = ds.pixel_array.astype(np.float32)
        if str(getattr(ds, 'PhotometricInterpretation', '')).upper() == 'MONOCHROME1':
            a = a.max()-a
        im = Image.fromarray(((a-a.min())/max(float(a.max()-a.min()), 1e-6)*255).astype(np.uint8))
    if im.mode in ('I;16', 'I;16B', 'I;16L', 'I', 'F'):          # 16-bit PNG -> 8-bit like the MIMIC JPGs
        a = np.asarray(im, np.float32); im = Image.fromarray(((a-a.min())/max(float(a.max()-a.min()), 1e-6)*255).astype(np.uint8))
    return im.convert('RGB')

def fetch(file_table, file_id, key):
    if IMAGE_ROOT:
        for pre in ['', 'train/', 'valid/', 'test/']:
            for ext in ['.jpg', '.png']:
                p = Path(IMAGE_ROOT)/(pre+key+ext)
                if p.is_file():
                    return p.read_bytes()
    return read_file_bytes(file_table, file_id)

def encode_all():
    fr = COHORT['frame']; n = len(fr)
    torch, model, preprocess, tok, txt_model, maxtok, dev, commit = load_encoders()
    # ---- decoding test on three images before the bulk run
    for i in range(3):
        im = decode(fetch(fr.file_table[i], fr.file_id[i], fr.key[i]))
        print(f'  decode test {i+1}/3 ok: {im.size} {im.mode}')
    jobs = [(i, 'f', fr.file_table[i], fr.file_id[i], fr.key[i]) for i in range(n)]
    jobs += [(i, 'l', fr.lat_file_table[i], fr.lat_file_id[i], fr.lat_key[i]) for i in range(n) if isinstance(fr.lat_key[i], str)]
    def one(job):
        i, kind, tn, fid, key = job
        try:
            return job, preprocess(decode(fetch(tn, fid, key)))
        except Exception as exc:
            return job, exc
    # cache is keyed by IMAGE (not by row), so re-runs and a larger N_CXP reuse every image already encoded
    cache = {}
    for part in sorted(PART_DIR.glob('img*.npz')):
        zz = read_npz(part); cache.update(zip(zz['key'].tolist(), zz['feats']))
    todo = [j for j in dict(((j[4], j) for j in jobs)).values() if j[4] not in cache]
    print(f'Images: {len(jobs):,} needed, {len(jobs)-len(todo):,} already cached, {len(todo):,} to encode', flush=True)
    t0 = time.time(); fails = []; nxt = len(list(PART_DIR.glob('img*.npz')))
    for c0 in range(0, len(todo), CHUNK):
        with cf.ThreadPoolExecutor(THREADS) as ex:
            res = list(ex.map(one, todo[c0:c0+CHUNK]))
        ok = [(j, x) for j, x in res if not isinstance(x, Exception)]
        fails += [(j[4], str(x)[:200]) for j, x in res if isinstance(x, Exception)]
        if ok:
            with torch.inference_mode():
                feats = np.concatenate([model.encode_image(torch.stack([x for _, x in ok[s:s+32]]).to(dev)).float().cpu().numpy() for s in range(0, len(ok), 32)])
            atomic_npz(PART_DIR/f'img{nxt:05d}.npz', key=np.array([j[4] for j, _ in ok], str), feats=feats.astype(np.float32)); nxt += 1
            cache.update(zip([j[4] for j, _ in ok], feats.astype(np.float32)))
        done = min(c0+CHUNK, len(todo)); rate = done/max(time.time()-t0, 1)
        print(f'  images {done:,}/{len(todo):,}  ({time.time()-t0:.0f}s, {rate:.1f}/s, ~{(len(todo)-done)/max(rate, 1e-6)/60:.0f} min left, failures {len(fails)})', flush=True)
    if fails:
        json_write(CXP_DIR/'encode_failures_PRIVATE.json', fails[:500])
    img = np.zeros((n, 512), np.float32); lat = np.zeros((n, 512), np.float32); okf = np.zeros(n, bool); okl = np.zeros(n, bool)
    for i, kind, _, _, key in jobs:
        if key in cache:
            if kind == 'f':
                img[i] = cache[key]; okf[i] = True
            else:
                lat[i] = cache[key]; okl[i] = True
    # ---- context text (same tokenizer settings and pooling as MIMIC)
    texts = fr.context.fillna('').astype(str).tolist(); present = np.array([bool(s.strip()) for s in texts])
    txt = np.zeros((n, txt_model.config.hidden_size), np.float32); idx = np.flatnonzero(present)
    with torch.inference_mode():
        for s in range(0, len(idx), 32):
            ii = idx[s:s+32]
            enc = tok([texts[i] for i in ii], padding=True, truncation=True, max_length=maxtok, return_tensors='pt').to(dev)
            h = txt_model(**enc).last_hidden_state; m_ = enc['attention_mask'].unsqueeze(-1)
            txt[ii] = ((h*m_).sum(1)/m_.sum(1).clamp_min(1)).float().cpu().numpy()
    keep = okf
    print(f'Encoded frontal {okf.sum():,}/{n:,}; lateral {okl.sum():,}; context present {present.mean():.1%}')
    extra = dict(y2=COHORT['y2'][keep], mask2=COHORT['mask2'][keep]) if 'y2' in COHORT else {}
    atomic_npz(CXP_NPZ, image=img[keep], lateral=lat[keep], eligible=okl[keep], text=txt[keep], present=present[keep],
               y=COHORT['y'][keep], mask=COHORT['mask'][keep], patient=fr.patient.to_numpy(str)[keep],
               sex=fr.sex_g.astype(str).to_numpy(str)[keep], age=fr.age_v.to_numpy(float)[keep], race=fr.race_g.to_numpy(str)[keep],
               has2=np.array('y2' in COHORT), biomedclip_commit=np.array(str(commit)), n_target=np.array(N_CXP), **extra)
    print('Saved', CXP_NPZ)

def _npz_ok():
    if not CXP_NPZ.is_file():
        return False
    with np.load(CXP_NPZ) as zz:
        return 'n_target' in zz.files and int(zz['n_target']) == N_CXP
if _npz_ok():
    print('Embeddings already cached:', CXP_NPZ)
else:
    encode_all()

## 6. External evaluation
1. Transfer AUROC for each label, and what clinical history and the lateral view add at the new hospital.
2. Risk targets with the MIMIC calibration unchanged, and after recalibrating only the threshold on one third of CheXpert Plus patients. Run twice: once with context (as available) and once image-only.
3. Escalations for the full gate versus the uncertainty, entropy, conformal and learning-to-defer gates, with Holm correction.
4. ASK-CONTEXT and ASK-LATERAL at the primary review budget, using the primary scenario: 30% of context withheld at triage, chosen label-blind.
5. Label-source sensitivity: labels from the whole report instead of the impression.
6. Subgroups by sex, age and race.

In [ ]:
# ---------------------------------------------------------------- EXTERNAL EVALUATION on CheXpert Plus (frozen MIMIC models, no retraining)
from sklearn.metrics import cohen_kappa_score
X_OUT = OUT/'external_chexpert_plus_v20'; X_OUT.mkdir(exist_ok=True)
GATES_X = ['entropy', 'ambiguity', 'L2D_U', 'L2D_UDMS', 'U', 'UDMS']
CAL_SCHEME = {'natural': 'natural', PRIMARY: PRIMARY, 'image_only': 'mcar60'}

def ext_score(sc, gate, kind):
    if gate in ('entropy', 'ambiguity'):
        return sc[gate]
    if gate.startswith('L2D_'):
        return sc['l2d_'+gate[4:]]
    return sc[('fn_' if kind == 'fn' else 'sv_')+gate]

def cal_score(cal, gate, kind):
    if gate in ('entropy', 'ambiguity'):
        return cal['sc_'+gate]
    if gate.startswith('L2D_'):
        return cal['sc_l2d_'+gate[4:]]
    return cal[('sc_fn_' if kind == 'fn' else 'sc_sv_')+gate]

def macro_auc(yy, pp, mm, sel, min_each=10):
    """Macro AUROC over assessable labels; sel is a boolean mask or an index array (bootstrap draws keep duplicates)."""
    idx = np.flatnonzero(sel) if np.asarray(sel).dtype == bool else np.asarray(sel)
    vals = []
    for j in range(len(LABELS)):
        k_ = idx[mm[idx, j] > .5]; t = yy[k_, j]
        if t.sum() >= min_each and len(t)-t.sum() >= min_each:
            vals.append(roc_auc_score(t, pp[k_, j]))
    return (float(np.mean(vals)) if vals else np.nan), len(vals)

def cxp_external():
    z = read_npz(CXP_NPZ)
    img_f, lat_f, txt_f = z['image'], z['lateral'], z['text']
    present, el = z['present'].astype(bool), z['eligible'].astype(bool)
    y, m = z['y'], z['mask']; pat = z['patient'].astype(str); n = len(pat)
    tz = (txt_f*present[:, None]).astype(np.float32)
    cal_i = hash_uniform(pat, 'cxpsplit') < 1/3; te_i = ~cal_i
    mfrac = int(re.sub(r'\D', '', PRIMARY))/100
    SCEN = {'natural': present, PRIMARY: present & (hash_uniform(pat, 'cxpctx') >= mfrac), 'image_only': np.zeros(n, bool)}
    folds = []
    for k in range(K):
        with (OUT/f'fold{k}_bundle_PRIVATE.pkl').open('rb') as h:
            B = pickle.load(h)
        H, T_ = B['H'], B['TEMP']
        raw = dict(img=H['img'].predict(img_f), ic=H['ic'].predict(np.hstack([img_f, tz])), txt=H['txt'].predict(tz))
        raw['il'] = raw['img'].copy(); raw['icl'] = raw['ic'].copy()
        if el.any():
            raw['il'][el] = H['il'].predict(np.hstack([img_f[el], lat_f[el]]))
            raw['icl'][el] = H['icl'].predict(np.hstack([img_f[el], tz[el], lat_f[el]]))
        pr = {key: apply_temperature(raw[key], T_[key]) for key in raw}
        per = {}
        for sname, avail in SCEN.items():
            p0 = np.where(avail[:, None], pr['ic'], pr['img']); pc = np.where(present[:, None], pr['ic'], pr['img'])
            pl = np.where(avail[:, None], pr['icl'], pr['il'])
            bl = signal_blocks(p0, pr['img'], pr['txt'], avail, img_f, tz, B['aux'], B['q'])
            sc = dict(entropy=bl['U'][:, 14], ambiguity=bl['U'][:, 15])
            for F, groups in GATE_SETS.items():
                X = design(bl, groups)
                sc['fn_'+F] = B['reg']['fn_'+F].predict(X); sc['sv_'+F] = B['reg']['sv_'+F].predict(X)
                sc['c0_'+F] = np.maximum(B['reg']['c0_'+F].predict(X), 0.)
            for F in ['U', 'UDMS']:
                sc['l2d_'+F] = B['reg']['l2d_'+F].score(design(bl, GATE_SETS[F]))
            Xf = design(bl, GATE_SETS['UDMS']); sc['gc'] = B['reg']['gc'].predict(Xf); sc['gl'] = B['reg']['gl'].predict(Xf)
            per[sname] = dict(avail=avail, ctx_ok=present & ~avail, p0=p0, pc=pc, pl=pl, o0=outcome(p0, y, m), oc=outcome(pc, y, m), ol=outcome(pl, y, m), sc=sc)
        folds.append(dict(pr=pr, per=per))
        print(f'  fold {k} applied', flush=True)
    R = dict(patients=int(n), context_present=float(present.mean()), lateral_available=float(el.mean()), cal_patients=int(cal_i.sum()), test_patients=int(te_i.sum()))

    # ---- 1) transfer quality and value of information
    pm = {key: np.mean([f['pr'][key] for f in folds], axis=0) for key in ['img', 'ic', 'il', 'icl']}
    rows = []
    for j, lab in enumerate(LABELS):
        for key, sel in [('img', np.ones(n, bool)), ('ic', present), ('il', el), ('icl', el & present)]:
            k_ = sel & (m[:, j] > .5); t = y[k_, j]
            if t.sum() >= 10 and len(t)-t.sum() >= 10:
                rows.append(dict(label=lab, input=key, n=int(k_.sum()), prevalence=float(t.mean()), auroc=float(roc_auc_score(t, pm[key][k_, j]))))
    AUC = pd.DataFrame(rows); TABLES['cxp_transfer_auroc'] = AUC
    display(AUC.pivot_table(index='label', columns='input', values='auroc').round(3))
    voi = []
    rng_ = np.random.default_rng(81)
    for a, b, sel, name in [('img', 'ic', present, 'clinical history'), ('img', 'il', el, 'lateral view'), ('ic', 'icl', el & present, 'lateral given history')]:
        A, na = macro_auc(y, pm[a], m, sel); Bm, nb = macro_auc(y, pm[b], m, sel)
        idx_ = np.flatnonzero(sel); d = []
        for _ in range(300):
            s_ = rng_.choice(idx_, len(idx_))
            d.append(macro_auc(y, pm[b], m, s_)[0]-macro_auc(y, pm[a], m, s_)[0])
        lo_, hi_ = np.nanquantile(d, [.025, .975])
        voi.append(dict(information=name, patients=int(sel.sum()), labels=na, macro_auroc_without=A, macro_auroc_with=Bm, delta=Bm-A, lo=float(lo_), hi=float(hi_)))
    VOI = pd.DataFrame(voi); TABLES['cxp_value_of_information'] = VOI; display(VOI.round(4))
    R['macro_auroc_image'] = macro_auc(y, pm['img'], m, np.ones(n, bool))[0]
    R['macro_auroc_context_as_available'] = macro_auc(y, np.where(present[:, None], pm['ic'], pm['img']), m, np.ones(n, bool))[0]

    # ---- 2) risk targets: MIMIC calibration unchanged vs threshold recalibrated on CheXpert Plus (same absolute targets as MIMIC)
    R['answer_all'] = {s: dict(fn=float(folds[0]['per'][s]['o0']['fn'][te_i].mean()), severe=float(folds[0]['per'][s]['o0']['severe'][te_i].mean())) for s in ['natural', 'image_only']}
    print('Answer-everything risk (test split):', R['answer_all'])
    rows = []; store = {}
    for sname in ['natural', 'image_only']:
        for kind, alphas in PLAN['harm_kinds'].items():
            hk = 'fn' if kind == 'fn' else 'severe'; h0 = 'fn0' if kind == 'fn' else 'sev0'
            for alpha in alphas:
                for gate in GATES_X:
                    lo_l, ln_l, eo_l, en_l = [], [], [], []
                    for k in range(K):
                        Pk = folds[k]['per'][sname]; s_ext = ext_score(Pk['sc'], gate, kind); harm = Pk['o0'][hk]
                        cal = get_cal(k, CAL_SCHEME[sname]); keep = ~cal['avail'] if sname == 'image_only' else np.ones(len(cal[h0]), bool)
                        t_old, _ = calibrate_gate(cal_score(cal, gate, kind)[keep], cal[h0][keep], alpha, PLAN['delta'], 'ltt')
                        t_new, _ = calibrate_gate(s_ext[cal_i], harm[cal_i], alpha, PLAN['delta'], 'ltt')
                        e_o, e_n = s_ext[te_i] >= t_old, s_ext[te_i] >= t_new
                        lo_l.append(harm[te_i]*~e_o); ln_l.append(harm[te_i]*~e_n); eo_l.append(e_o); en_l.append(e_n)
                    lo_, ln_ = np.mean(lo_l, axis=0), np.mean(ln_l, axis=0); eo_, en_ = np.mean(eo_l, axis=0), np.mean(en_l, axis=0)
                    store[(sname, kind, alpha, gate)] = (ln_, lo_, en_, eo_)
                    ci_o, ci_n = boot_ci(lo_, PLAN['boot'], 91), boot_ci(ln_, PLAN['boot'], 92)
                    rows.append(dict(scenario=sname, harm=kind, alpha=alpha, gate=gate,
                                     referral_mimic_calibrated=float(eo_.mean()), risk_mimic_calibrated=ci_o['mean'], risk_hi_mimic=ci_o['hi'], target_met_mimic=bool(ci_o['mean'] <= alpha),
                                     referral_cxp_recalibrated=float(en_.mean()), risk_cxp_recalibrated=ci_n['mean'], risk_hi_cxp=ci_n['hi'], target_met_cxp=bool(ci_n['mean'] <= alpha)))
    G = pd.DataFrame(rows); TABLES['cxp_gate'] = G
    display(G[G.gate.isin(['U', 'UDMS'])].round(4))
    for sname in ['natural', 'image_only']:
        fu = G[(G.gate == 'UDMS') & (G.scenario == sname)]
        R[f'targets_{sname}'] = dict(configs=int(len(fu)), met_mimic_calibrated=int(fu.target_met_mimic.sum()), met_cxp_recalibrated=int(fu.target_met_cxp.sum()),
                                     upper_bound_met_cxp=int((fu.risk_hi_cxp <= fu.alpha).sum()))
    # ---- 3) escalation contrasts (recalibrated thresholds, context as available), Holm within the family
    cons = []
    pairs = [('UDMS', 'U'), ('U', 'entropy'), ('U', 'ambiguity'), ('U', 'L2D_U'), ('UDMS', 'L2D_UDMS')]
    for kind, alphas in PLAN['harm_kinds'].items():
        for alpha in alphas:
            for a, b in pairs:
                ci = boot_ci(store[('natural', kind, alpha, a)][2]-store[('natural', kind, alpha, b)][2], PLAN['boot'], 93)
                cons.append(dict(harm=kind, alpha=alpha, comparison=f'{a} vs {b}', delta_referral=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'], mde80=ci['mde80']))
    C = pd.DataFrame(cons); C['p_holm'] = np.nan
    for comp, idx_ in C.groupby('comparison').groups.items():
        C.loc[idx_, 'p_holm'] = holm(C.loc[idx_, 'p'].to_numpy())
    C['verdict_holm'] = np.where(C.p_holm < .05, np.where(C.delta_referral < 0, 'better', 'worse'), 'inconclusive')
    TABLES['cxp_contrasts'] = C; display(C.round(4))
    pe = C[(C.harm == PLAN['primary_endpoint']['harm']) & (C.alpha == PLAN['primary_endpoint']['alpha'])]
    R['primary_endpoint_contrasts'] = pe.to_dict('records')

    # ---- 4) ASK-CONTEXT / ASK-LATERAL at the primary review budget (primary scenario: context withheld for some patients at triage)
    p_none = dict(P_ASK, allow_ctx=False, allow_lat=False)
    VAR = {'ask_context': dict(P_ASK, allow_ctx=True, allow_lat=False), 'ask_lateral': dict(P_ASK, allow_ctx=False, allow_lat=True),
           'ask_both': dict(P_ASK, allow_ctx=True, allow_lat=True)}
    def ask_delta(pv, budget):
        d, rc, rl = [], [], []
        for k in range(K):
            Pk = folds[k]['per'][PRIMARY]; o0, oc, ol, sc = Pk['o0'], Pk['oc'], Pk['ol'], Pk['sc']
            c0 = o0['fn']+FPW*o0['fp']; cc = oc['fn']+FPW*oc['fp']; cl = ol['fn']+FPW*ol['fp']
            av = pick_lambda4(sc['c0_UDMS'], sc['gc'], sc['gl'], Pk['ctx_ok'], el, EXPERT_E, pv, budget, LAM)[1]
            a0 = pick_lambda4(sc['c0_UDMS'], sc['gc'], sc['gl'], Pk['ctx_ok'], el, EXPERT_E, p_none, budget, LAM)[1]
            d.append(realised4(av, c0, cc, cl, EXPERT_E, pv)-realised4(a0, c0, cc, cl, EXPERT_E, p_none)); rc.append((av == 2).mean()); rl.append((av == 3).mean())
        return np.mean(d, axis=0), float(np.mean(rc)), float(np.mean(rl))
    rows = []
    for name, pv in VAR.items():
        dd, rc, rl = ask_delta(pv, PLAN['primary_budget']); ci = boot_ci(dd, PLAN['boot'], 94)
        rows.append(dict(action=name, budget=PLAN['primary_budget'], a_c=pv['a_c'], a_l=pv['a_l'], delta_cost=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'],
                         verdict=verdict(ci), ask_context_rate=rc, ask_lateral_rate=rl))
    for a_c in PLAN['ctx_grid_a']:                          # break-even: how cheap must a history request be?
        pv = dict(VAR['ask_context'], a_c=a_c); dd, rc, rl = ask_delta(pv, PLAN['primary_budget']); ci = boot_ci(dd, PLAN['boot'], 95)
        rows.append(dict(action='ask_context_grid', budget=PLAN['primary_budget'], a_c=a_c, a_l=pv['a_l'], delta_cost=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'],
                         verdict=verdict(ci), ask_context_rate=rc, ask_lateral_rate=rl))
    for a_l in PLAN['lat_grid_a']:
        pv = dict(VAR['ask_lateral'], a_l=a_l); dd, rc, rl = ask_delta(pv, PLAN['primary_budget']); ci = boot_ci(dd, PLAN['boot'], 96)
        rows.append(dict(action='ask_lateral_grid', budget=PLAN['primary_budget'], a_c=pv['a_c'], a_l=a_l, delta_cost=ci['mean'], lo=ci['lo'], hi=ci['hi'], p=ci['p'],
                         verdict=verdict(ci), ask_context_rate=rc, ask_lateral_rate=rl))
    ASK = pd.DataFrame(rows); TABLES['cxp_ask'] = ASK; display(ASK.round(5))
    R['ask'] = ASK[ASK.action.isin(list(VAR))].to_dict('records')

    # ---- 5) second labeler (CheXbert), if available
    if bool(z['has2']):
        y2, m2 = z['y2'], z['mask2']; kr = []
        for j, lab in enumerate(LABELS):
            b_ = (m[:, j] > .5) & (m2[:, j] > .5)
            if b_.sum() >= 50 and len(np.unique(y[b_, j])) == 2 and len(np.unique(y2[b_, j])) == 2:
                kr.append(dict(label=lab, n=int(b_.sum()), kappa=float(cohen_kappa_score(y[b_, j], y2[b_, j])), prev_primary=float(y[b_, j].mean()), prev_second=float(y2[b_, j].mean())))
        KAP = pd.DataFrame(kr); TABLES['cxp_second_labeler_kappa'] = KAP; display(KAP.round(3))
        rows = []
        for kind, alphas in PLAN['harm_kinds'].items():
            hk = 'fn' if kind == 'fn' else 'severe'
            for alpha in alphas:
                for gate in ['U', 'UDMS']:
                    keep_l, recal_l, ek, er = [], [], [], []
                    for k in range(K):
                        Pk = folds[k]['per']['natural']; s_ = ext_score(Pk['sc'], gate, kind)
                        h1 = Pk['o0'][hk]; h2 = outcome(Pk['p0'], y2, m2)[hk]
                        t1, _ = calibrate_gate(s_[cal_i], h1[cal_i], alpha, PLAN['delta'], 'ltt'); t2, _ = calibrate_gate(s_[cal_i], h2[cal_i], alpha, PLAN['delta'], 'ltt')
                        e1, e2 = s_[te_i] >= t1, s_[te_i] >= t2
                        keep_l.append(h2[te_i]*~e1); recal_l.append(h2[te_i]*~e2); ek.append(e1); er.append(e2)
                    ck, cr = boot_ci(np.mean(keep_l, axis=0), PLAN['boot'], 97), boot_ci(np.mean(recal_l, axis=0), PLAN['boot'], 98)
                    rows.append(dict(harm=kind, alpha=alpha, gate=gate, risk_keep_primary_calibration=ck['mean'], hi_keep=ck['hi'], met_keep=bool(ck['mean'] <= alpha),
                                     risk_recalibrated_second=cr['mean'], hi_recal=cr['hi'], met_recal=bool(cr['mean'] <= alpha),
                                     referral_keep=float(np.mean(ek)), referral_recal=float(np.mean(er))))
        SL = pd.DataFrame(rows); TABLES['cxp_second_labeler_risk'] = SL; display(SL.round(4))
        fu = SL[SL.gate == 'UDMS']
        R['second_labeler'] = dict(table=M.get('second_label_table') if 'M' in globals() else None, macro_kappa=float(KAP.kappa.mean()) if len(KAP) else None,
                                   configs=int(len(fu)), met_keep=int(fu.met_keep.sum()), met_recal=int(fu.met_recal.sum()))
    else:
        print('No alternative label source found; label-source check skipped.')

    # ---- 6) subgroups (context as available, recalibrated thresholds)
    sex, age, race = z['sex'].astype(str), z['age'].astype(float), z['race'].astype(str)
    groups = {'sex=F': sex == 'F', 'sex=M': sex == 'M', 'age<40': age < 40, 'age 40-64': (age >= 40) & (age < 65), 'age>=65': age >= 65}
    for r_ in sorted(set(race)):
        groups['race='+r_] = race == r_
    p0m = np.mean([f['per']['natural']['p0'] for f in folds], axis=0)
    rows = []
    for (sname, kind, alpha, gate), (ln_i, lo_i, en_i, eo_i) in store.items():
        if sname != 'natural' or gate not in ('U', 'UDMS'):
            continue
        for gname, mg_all in groups.items():
            mg = mg_all[te_i]
            if mg.sum() < 100:
                continue
            ci = boot_ci(ln_i[mg], PLAN['boot'], 99)
            au, nl = macro_auc(y[te_i], p0m[te_i], m[te_i], mg)
            rows.append(dict(harm=kind, alpha=alpha, gate=gate, group=gname, n=int(mg.sum()), macro_auroc=au, auroc_labels=nl,
                             answer_all_risk=float(folds[0]['per']['natural']['o0']['fn' if kind == 'fn' else 'severe'][te_i][mg].mean()),
                             risk=ci['mean'], lo=ci['lo'], hi=ci['hi'], referral=float(en_i[mg].mean()),
                             point_target_met=bool(ci['mean'] <= alpha), upper_bound_met=bool(ci['hi'] <= alpha)))
    SG = pd.DataFrame(rows); TABLES['cxp_subgroups'] = SG
    if len(SG):
        display(SG[(SG.gate == 'UDMS')].round(4))
        fu = SG[SG.gate == 'UDMS']
        R['subgroups'] = dict(groups=sorted(fu.group.unique().tolist()), configs=int(len(fu)), point_target_met=int(fu.point_target_met.sum()),
                              upper_bound_met=int(fu.upper_bound_met.sum()), exceedances=fu[~fu.point_target_met][['group', 'harm', 'alpha', 'risk']].to_dict('records'))
    RESULTS['external_chexpert_plus'] = R
    print(json.dumps({k_: v_ for k_, v_ in R.items() if k_ not in ('ask', 'primary_endpoint_contrasts')}, indent=1, default=str))

TABLES.clear(); RESULTS.clear()
cxp_external()

## 7. Share-safe export
CSV tables, a JSON summary and a zip. No patient-level arrays, embeddings, report text or models.

In [ ]:
import zipfile
for name, df in TABLES.items():
    df.to_csv(X_OUT/f'{name}.csv', index=False)
json_write(X_OUT/'results_v20_chexpert_plus.json', RESULTS)
try:
    with pd.ExcelWriter(X_OUT/'master_v20_chexpert_plus.xlsx') as xw:
        for name, df in TABLES.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
except Exception as exc:
    print('Excel export skipped:', exc)
zp = X_OUT/'V20_CHEXPERT_PLUS_RESULTS.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z_:
    for p in sorted(X_OUT.iterdir()):
        if p.is_file() and 'PRIVATE' not in p.name and p.suffix in ('.csv', '.json', '.xlsx'):
            z_.write(p, p.name)
    sch = CXP_DIR/'schema_summary.json'
    if sch.is_file():
        z_.write(sch, sch.name)
print('Share-safe bundle:', zp)
try:
    from google.colab import files
    files.download(str(zp))
except Exception:
    pass

## How to read the outcome (decide before looking)
* **Targets met with the MIMIC calibration unchanged, with context available.** The guarantee transfers to a second hospital in the full multimodal setting. This is stronger than NIH, which was image-only.
* **Targets met only after recalibrating the threshold.** Report that the site-specific threshold is needed. This is consistent with the exchangeability caveat.
* **Clinical history raises AUROC but ASK-CONTEXT is inconclusive.** This replicates the MIMIC finding at another site: information value does not equal routing value.
* **Subgroup exceedances.** Report them as in MIMIC. The guarantee is marginal.

Send me the zip (or the printed tables). I'll then update the manuscript: abstract, section 3.2, the subgroup paragraph, limitations, Fig. 1, Fig. 3 and Fig. S2.